In [1]:
import os
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import torch
import timm
from torchvision.io import read_image, ImageReadMode
import torch.nn.functional as F
import numpy as np
import torch.optim as optim
from torch import nn
from collections import Counter



c:\Users\guilh\Git\Elifoot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def split_data(data_list, train_proportion=0.7, test_proportion=0.2):
    data_list_len = len(data_list)
    shuffled_data = list(np.random.permutation(data_list))

    train_size = int(np.ceil(data_list_len * train_proportion))
    test_size = int(np.ceil(data_list_len * test_proportion))
    
    return {'train': shuffled_data[0: train_size], 
            'test': shuffled_data[train_size: train_size + test_size], 
            'val': shuffled_data[train_size + test_size:]}


In [3]:
class EliImageDataset(Dataset):
    def __init__(self, img_path_list, all_classes_df=None, transform=None):

        self.img_path_list = img_path_list
        self.img_path_df = pd.Series(self.img_path_list).explode().reset_index()
        self.img_path_df.columns = ['Label', 'Img_path'] 

        if all_classes_df is not None:
            self.all_classes_df = all_classes_df
        else:
            cd = Counter([i.split('\\')[-2] for i in self.img_path_list])
            classes_df = pd.DataFrame(cd, index=[0]).transpose().reset_index()
            classes_df.columns = ['Class', 'Count']
            classes_df['Label'] = classes_df.index
            self.all_classes_df = classes_df

        if not transform:
            self.transform = v2.Compose([
                v2.ToImage(),             
                v2.ToDtype(torch.float32, scale=True),
                v2.Resize((120, 120), antialias=True)
                ])
        else:
            self.transform = transform


    def __len__(self):
        return len(self.img_path_list)

    def __getitem__(self, idx):
        image = read_image(self.img_path_df.iloc[idx]['Img_path'], mode=ImageReadMode.RGB)
        label = self.img_path_df.iloc[idx]['Img_path'].split('\\')[2]

        if self.transform:
            image = self.transform(image)

        label = self.all_classes_df.loc[self.all_classes_df['Class'] == label, 'Label'].item()
        return image, label

In [4]:
img_labels_folder = r'Data\Frames_Categories'
img_path_dict = {}
for folder in os.listdir(img_labels_folder):
    if folder == 'Other':
        continue
    folder_path = os.path.join(img_labels_folder, folder)
    img_path_dict[folder] =[os.path.join(folder_path, img_name) for img_name in  os.listdir(folder_path)]

In [5]:
imgs_split_dict = {split_type: [] for split_type in ['train', 'test', 'val']}

for img_class in img_path_dict.keys():
    class_split_dict = split_data(img_path_dict[img_class])
    for split_type in class_split_dict.keys(): 
        imgs_split_dict[split_type] += class_split_dict[split_type]

In [6]:
ds_train = EliImageDataset(imgs_split_dict['train'])
ds_val = EliImageDataset(imgs_split_dict['val'], ds_train.all_classes_df)
ds_test = EliImageDataset(imgs_split_dict['test'], ds_train.all_classes_df)

In [7]:
train_dataloader = DataLoader(ds_train, batch_size=64, shuffle=True)
val_dataloader = DataLoader(ds_val, batch_size=64)
test_dataloader = DataLoader(ds_test, batch_size=64)

In [8]:
1/0

ZeroDivisionError: division by zero

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [10]:
model = timm.create_model('convnext_tiny', pretrained=True, num_classes=len(set([i.split('\\')[-2] for i in imgs_split_dict['train']])))

In [42]:
torch.tensor(ds_train.all_classes_df['Count'] / ds_train.all_classes_df['Count'].sum()).float()

tensor([0.0054, 0.0889, 0.0836, 0.1402, 0.2022, 0.0243, 0.1024, 0.0296, 0.0081,
        0.1456, 0.1213, 0.0431, 0.0054])

In [50]:
w = torch.tensor([10, 0.0889, 0.0836, 0.1402, 0.2022, 0.0243, 0.1024, 0.0296, 0.0081,
        0.1456, 0.1213, 0.0431, 10]).float()

In [51]:
criterion = torch.nn.CrossEntropyLoss(w)
optimizer = optim.AdamW(model.parameters(), lr=0.1, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode='max',
                                                       patience=1, factor=0.4)
softmax = nn.Softmax(dim=1)

In [52]:
all_losses = []
d = {}

for epoch in range(10):
    epoch_loss = []
    epoch_preds = []
    for img, labels in train_dataloader:


        outputs = model.to(device)(img.to(device))
        loss = criterion(outputs, labels.to(device))
        print(loss)
        #epoch_loss += [loss.item()]

        preds = softmax(outputs).max(axis=1).indices.tolist()
        epoch_loss += [preds == labels.tolist()]

        optimizer.zero_grad()


        loss.backward()
        optimizer.step()
        epoch_preds += preds
        
    epoch_mean_loss_train = np.mean(epoch_loss)
    d[epoch] = epoch_preds

    all_preds_val = []
    all_labels_val = []

    for img_val, label_val in test_dataloader:
        with torch.no_grad():
            outputs = model.to(device)(img_val)
            all_preds_val += softmax(outputs).max(axis=1).indices.tolist()
            all_labels_val += label_val.tolist()

    preds_df = pd.DataFrame([all_preds_val, all_labels_val]).transpose()
    preds_df.columns = ['Pred', 'Label']
    
    epoch_mean_loss = np.mean(preds_df['Pred'] == preds_df['Label'])
    print(f'Epoch {epoch}, Train:{epoch_mean_loss_train}, Val:{epoch_mean_loss}')
 


tensor(6.6695, grad_fn=<NllLossBackward0>)
tensor(21.5084, grad_fn=<NllLossBackward0>)
tensor(15.2775, grad_fn=<NllLossBackward0>)
tensor(15.1573, grad_fn=<NllLossBackward0>)
tensor(22.0709, grad_fn=<NllLossBackward0>)
tensor(38.8695, grad_fn=<NllLossBackward0>)
Epoch 0, Train:0.0, Val:0.12149532710280374
tensor(40.6489, grad_fn=<NllLossBackward0>)
tensor(22.5867, grad_fn=<NllLossBackward0>)
tensor(24.6032, grad_fn=<NllLossBackward0>)
tensor(16.9758, grad_fn=<NllLossBackward0>)
tensor(7.2056, grad_fn=<NllLossBackward0>)
tensor(19.3870, grad_fn=<NllLossBackward0>)
Epoch 1, Train:0.0, Val:0.0
tensor(16.5802, grad_fn=<NllLossBackward0>)
tensor(4.1796, grad_fn=<NllLossBackward0>)
tensor(7.2120, grad_fn=<NllLossBackward0>)
tensor(12.4408, grad_fn=<NllLossBackward0>)
tensor(7.0915, grad_fn=<NllLossBackward0>)
tensor(5.1153, grad_fn=<NllLossBackward0>)
Epoch 2, Train:0.0, Val:0.14018691588785046
tensor(4.2105, grad_fn=<NllLossBackward0>)
tensor(7.4627, grad_fn=<NllLossBackward0>)
tensor(2.956

In [ ]:
ds_test = EliImageDataset(imgs_split_dict['train'])
test_dataloader = DataLoader(ds_test, batch_size=64, shuffle=False)

In [ ]:
all_preds = []
all_labels = []

for img, labels in test_dataloader:
     with torch.no_grad():
          outputs = model.to(device)(img)
          all_preds += softmax(outputs).max(axis=1).indices.tolist()
          all_labels += labels.tolist()